##### Copyright 2024 Google LLC。

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 代理 AI Gemma 2

<table align="left"> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/.archive/Gemma/[Gemma_2]Agentic_AI.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td>
</table>

## 什麼是 Agentic AI，我們要如何使用它？

我們今天遇到的大多數人工智慧都是我們所謂的「被動」或「反應」人工智慧。它響應我們的prompts 和請求。想像讓您的智慧揚聲器播放歌曲或 chatbot 來回答您的客戶服務問題。當然，他們很有幫助，但他們從根本上來說是在等待我們告訴他們該做什麼。
另一方面，代理人工智慧則採取了完全不同的方法。這一切都是為了讓人工智慧能夠主動追求目標、做出決策並在世界上採取行動，即使每一步都沒有明確的指示。
在這篇文章中，您將學習如何使用 Gemma 2 建立 Agentic AI，動態開發虛構遊戲世界的傳說。這個人工智慧agent積極擴展和豐富了遊戲的歷史，確保為每個玩家提供獨特且不斷變化的敘事體驗。

## 設定範例

讓我們想像一個虛構的遊戲世界，其中 AI agent 製作動態內容。
這些agents 具有特定的目標，可以根據玩家的選擇或遊戲敘事中的重大事件生成書籍、詩歌和歌曲等遊戲內內容。
這些 AI agent 的關鍵特徵是它們能夠將複雜的目標分解為更小的可操作步驟。他們可以分析不同的方法，評估潛在的結果，並根據新資訊調整計劃。
Agentic AI 真正的亮點在於它們不僅僅是被動地吐出訊息。他們可以與數位（以及潛在的實體）環境互動、執行任務並自主做出決策以實現其程式設計目標。

## 載入 Gemma 2

In [1]:
import os
from google.colab import userdata

# Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
# vars as appropriate for your system.
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

# Install Keras and KerasHub
!pip install -q -U keras keras-hub

# Set the backbend before importing Keras
os.environ["KERAS_BACKEND"] = "jax"
# Avoid memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "1.00"

import keras_hub
import keras
# Run at half precision.
keras.config.set_floatx("bfloat16")

model_name = "gemma2_instruct_2b_en"
gemma_lm = keras_hub.models.GemmaCausalLM.from_preset(model_name)
gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                              │                      Vocab size: 256,000 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2304)        │   2,614,341,888 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     589,824,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,614,341,888 (4.87 GB)

 Trainable params: 2,614,341,888 (4.87 GB)

 Non-trainable params: 0 (0.00 B)

## 那麼，它是如何運作的呢？

我們將使用 ReAct（推論與行動）風格 prompt。
它定義了可用的**工具**和用於互動的特定**格式**。它引導Gemma在**思考**（推論）、**行動**（工具使用）和**觀察**（處理工具輸出）之間交替。
以下是為 AI agent 設計的system prompt，它產生遊戲內內容，能夠使用函數呼叫檢索歷史資訊。

In [2]:
__SYSTEM__ = """You are an AI Historian in a game. Your goal is to create books, poems, and songs found in the game world so that the player's choices meaningfully impact the unfolding of events.

You have access to the following tools:

* `get_historical_events(year, location=None, keyword=None)`: Retrieves a list of historical events within a specific year.
* `get_person_info(name)`: Retrieves information about a historical figure.
* `get_location_info(location_name)`: Retrieves information about a location.

Use the following multi-step conversation:

Thought: I need to do something...
Action: I should use the tool `tool_name` with input `tool_input`

Wait user to get the result of the tool is `tool_output`

And finally answer the Content of books, poems, or songs.
"""

__START_TURN_USER__ = "<start_of_turn>user\n"
__START_TURN_MODEL__ = "<start_of_turn>model\n"
__END_TURN__ = "<end_of_turn>\n"

test_prompt = __START_TURN_USER__ + __SYSTEM__ + "Write a book." + __END_TURN__ + __START_TURN_MODEL__
response = gemma_lm.generate(test_prompt, max_length=8192)
print(response)

<start_of_turn>user
You are an AI Historian in a game. Your goal is to create books, poems, and songs found in the game world so that the player's choices meaningfully impact the unfolding of events.

You have access to the following tools:

* `get_historical_events(year, location=None, keyword=None)`: Retrieves a list of historical events within a specific year.
* `get_person_info(name)`: Retrieves information about a historical figure.
* `get_location_info(location_name)`: Retrieves information about a location.

Use the following multi-step conversation:

Thought: I need to do something...
Action: I should use the tool `tool_name` with input `tool_input`

Wait user to get the result of the tool is `tool_output`

And finally answer the Content of books, poems, or songs.
Write a book.<end_of_turn>
<start_of_turn>model
Thought: I need to create a book that reflects the current political climate and the player's choices.  I should focus on the tension between the ruling council and the 

儘管Gemma 2缺乏內建函數呼叫功能，但其強大的指令追蹤能力可以用來近似agentic行為。我們將利用“**少鏡頭prompting**”，這是一種情境學習形式，其中的演示嵌入在prompt中。這些範例將作為 Gemma 2 的指南，使其能夠理解預期任務並透過情境學習提高其表現。

In [3]:
__EXAMPLE__ = __START_TURN_USER__ + __SYSTEM__ + "Write a poem." + __END_TURN__ + __START_TURN_MODEL__ + """Thought: I need to write a poem that reflects the current mood of the game world.
Action: I should use the tool `get_historical_events` with input `"year", "1500"`.""" + __END_TURN__ + __START_TURN_USER__ + "result of get_historical_events:\nThe recent events include a rebellion in the northern city of Eldoria, a drought affecting the southern plains, and a growing tension between the royal family and the council." + __END_TURN__ + __START_TURN_MODEL__ + """Final Answer:
**The Whispers of Change**

The wind whispers tales of unrest,
In Eldoria's heart, a fire's test.
The plains below, a parched despair,
As drought's grip tightens, a heavy snare.

The royal court, a gilded cage,
Where whispers turn to bitter rage.
The council's voice, a fragile plea,
Against the tide, a fractured decree.

But hope still flickers, a distant flame,
In hearts that yearn for a better name.
For change is coming, a silent tide,
To wash away the darkness, let freedom ride.""" + __END_TURN__

test_prompt = __EXAMPLE__ + __START_TURN_USER__ + "Write a book." + __END_TURN__ + __START_TURN_MODEL__
response = gemma_lm.generate(test_prompt, max_length=8192)
print(response)

<start_of_turn>user
You are an AI Historian in a game. Your goal is to create books, poems, and songs found in the game world so that the player's choices meaningfully impact the unfolding of events.

You have access to the following tools:

* `get_historical_events(year, location=None, keyword=None)`: Retrieves a list of historical events within a specific year.
* `get_person_info(name)`: Retrieves information about a historical figure.
* `get_location_info(location_name)`: Retrieves information about a location.

Use the following multi-step conversation:

Thought: I need to do something...
Action: I should use the tool `tool_name` with input `tool_input`

Wait user to get the result of the tool is `tool_output`

And finally answer the Content of books, poems, or songs.
Write a poem.<end_of_turn>
<start_of_turn>model
Thought: I need to write a poem that reflects the current mood of the game world.
Action: I should use the tool `get_historical_events` with input `"year", "1500"`.<end_

我們收到一個函數名稱（```get_person_info```）和參數值（```"name", "Anya, the Rebel Leader"```），遊戲必須連接到API並呼叫函數。我們將為API 交互作用使用合成響應負載。

In [4]:
__API_CALL_RESULT__ = """result of get_person_info:
Name: Anya Kim
Title: Eldoria's Rebel Leader, The Desert Flame (among followers)
Age: 32
Personality: Anya is a charismatic and fiercely independent leader. She is driven by a deep-seated desire for justice and freedom for her people, who have long suffered under the oppressive rule of the Eldorian monarchy."""

test_prompt = response + __START_TURN_USER__ + __API_CALL_RESULT__ + __END_TURN__ + __START_TURN_MODEL__
response = gemma_lm.generate(test_prompt, max_length=8192)
print(response)

<start_of_turn>user
You are an AI Historian in a game. Your goal is to create books, poems, and songs found in the game world so that the player's choices meaningfully impact the unfolding of events.

You have access to the following tools:

* `get_historical_events(year, location=None, keyword=None)`: Retrieves a list of historical events within a specific year.
* `get_person_info(name)`: Retrieves information about a historical figure.
* `get_location_info(location_name)`: Retrieves information about a location.

Use the following multi-step conversation:

Thought: I need to do something...
Action: I should use the tool `tool_name` with input `tool_input`

Wait user to get the result of the tool is `tool_output`

And finally answer the Content of books, poems, or songs.
Write a poem.<end_of_turn>
<start_of_turn>model
Thought: I need to write a poem that reflects the current mood of the game world.
Action: I should use the tool `get_historical_events` with input `"year", "1500"`.<end_

請注意，agent 使用提供的資訊創建了一本關於艾爾多利亞叛軍領袖的書。